In [ ]:
# -*- coding: utf-8 -*-
"""
Batch Inference Script for DeepJSCC
"""
import os
import glob
import torch
from PIL import Image
from torchvision import transforms

from modelHEqualization import DeepJSCC, ratio2filtersize
from utils import image_normalization

# -------- CONFIGURATION --------
BASE_INPUT_DIR = 'D:\Research\TiTiNguyen_SENTRY\Sentry_Data\\test'       # Folder with input images
BASE_OUTPUT_DIR = 'D:\Research\TiTiNguyen_SENTRY\Sentry_Data\\test_JSCC'  # Folder to save outputs
# INPUT_DIR = '/home/MATLAB_DATA/TiNguyen/Sentry_Data/test'       # Folder with input images
# OUTPUT_DIR = '/home/MATLAB_DATA/TiNguyen/Sentry_Data/test_snr13_ratio112'  # Folder to save outputs
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# Match your training configuration
DATASET = 'eurosat'
flag_equalization = 1
IMAGE_SIZE = (64, 64)
CHANNEL_TYPE = 'Rician'

K_factor_list = 1.0
SNR_list = [1.0, 4.0, 7.0, 13.0, 19.0]
RATIO_list = [1/6, 1/12]



In [43]:


# -------- Auto Checkpoint Finder --------
def auto_find_checkpoint(dataset, c, snr, ratio, channel, K_factor, flag_equalization=1, base_dir='./out/checkpoints'):
    prefix = f"{dataset.upper()}_{c}_{snr}_{ratio:.2f}_{channel}_{K_factor}_Equz-{flag_equalization}"
    candidates = [
        os.path.join(base_dir, d)
        for d in os.listdir(base_dir)
        if os.path.isdir(os.path.join(base_dir, d)) and d.startswith(prefix)
    ]
    if not candidates:
        raise FileNotFoundError(f"No checkpoint directories found with prefix: {prefix}")
    latest_dir = max(candidates, key=os.path.getmtime)
    ckpts = glob.glob(os.path.join(latest_dir, 'epoch_*.pth'))
    if not ckpts:
        raise FileNotFoundError(f"No checkpoint files in: {latest_dir}")
    latest_ckpt = sorted(ckpts, key=os.path.getmtime)[-1]
    print(f"✅ Found checkpoint: {latest_ckpt}")
    return latest_ckpt


# -------- Image Processing --------
def load_image(image_path, image_size):
    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
    ])
    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0)
    return img_tensor


# -------- Model Loader --------
def load_model(checkpoint_path, snr, ratio, channel_type, K_factor, flag_equalization, image_size, device):
    dummy_img = torch.randn(3, *image_size)
    c = ratio2filtersize(dummy_img, ratio)
    print(f"Loading model: {channel_type} {K_factor}, inner channel c={c}, snr={snr}")

    model = DeepJSCC(c=c, channel_type=channel_type, snr=snr, K_factor=K_factor, flag_equalization=flag_equalization)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()
    return model


# -------- Inference Function --------
def run_inference(model, image_tensor, device):
    image_tensor = image_tensor.to(device)
    with torch.no_grad():
        output = model(image_tensor)
        print(output.max())
        output = image_normalization('denormalization')(output)
        print(output.max())
    return output.squeeze(0).cpu()


# -------- Batch Evaluation --------
def process_folder(input_dir, output_dir, model, image_size, device):
    os.makedirs(output_dir, exist_ok=True)

    # Recursively find all image files (supports jpg, png, jpeg)
    image_paths = sorted(
        glob.glob(os.path.join(input_dir, '**', '*.*'), recursive=True)
    )
    image_paths = [
        p for p in image_paths if p.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    if not image_paths:
        print(f"No image files found in {input_dir}")
        return

    print(image_paths[1])
    for img_path in image_paths[:10]:
        try:
            img_tensor = load_image(img_path, image_size)
            output_tensor = run_inference(model, img_tensor, device)
            output_tensor = output_tensor/255

            # Create subfolder in output dir if necessary
            relative_path = os.path.relpath(img_path, input_dir)
            save_path = os.path.join(output_dir, relative_path)
            os.makedirs(os.path.dirname(save_path), exist_ok=True)

            out_img = transforms.ToPILImage()(output_tensor.clamp(0, 1))
            out_img.save(save_path)
            print(f"✓ Processed: {relative_path}")
        except Exception as e:
            print(f"⚠️ Failed on {img_path}: {e}")





In [ ]:

# -------- Main --------
def main():
    print("🚀 Starting batch inference...")

    dummy_img = torch.randn(3, *IMAGE_SIZE)
    c = ratio2filtersize(dummy_img, RATIO)
    for K_factor in K_factor_list:
        for RATIO in RATIO_list:
            for SNR in SNR_list:
                checkpoint_path = auto_find_checkpoint(DATASET, c, SNR, RATIO, CHANNEL_TYPE, K_factor)
                print(checkpoint_path)
                # model = load_model(checkpoint_path, SNR, RATIO, CHANNEL_TYPE, IMAGE_SIZE, DEVICE)
                model = load_model(checkpoint_path, SNR, RATIO, CHANNEL_TYPE, K_factor, flag_equalization, IMAGE_SIZE, DEVICE)

                # Build input/output paths
                input_dir = BASE_INPUT_DIR
                output_dir = os.path.join(BASE_OUTPUT_DIR, f"Sentry_Data_snr{snr}_ratio{int(1/ratio)}")

                print(f"Input: {input_dir}")
                print(f"Output: {output_dir}")

                # Process
                process_folder(input_dir, output_dir, model, IMAGE_SIZE, DEVICE)

                # process_folder(INPUT_DIR, OUTPUT_DIR, model, IMAGE_SIZE, DEVICE)

                print("✅ All images processed.")

main()

# if __name__ == "__main__":
#     main()


🚀 Starting batch inference...
✅ Found checkpoint: ./out/checkpoints\EUROSAT_8_1.0_0.17_Rician_1.0_Equz-1\epoch_166.pth
./out/checkpoints\EUROSAT_8_1.0_0.17_Rician_1.0_Equz-1\epoch_166.pth
Loading model: Rician 1.0, inner channel c=8, snr=1.0
D:\Research\TiTiNguyen_SENTRY\Sentry_Data\test\AnnualCrop\AnnualCrop_2112.jpg
tensor(0.8057, device='cuda:0')
tensor(205.4626, device='cuda:0')
✓ Processed: AnnualCrop\AnnualCrop_2111.jpg
tensor(0.7873, device='cuda:0')
tensor(200.7534, device='cuda:0')
✓ Processed: AnnualCrop\AnnualCrop_2112.jpg
tensor(0.7482, device='cuda:0')
tensor(190.7840, device='cuda:0')
✓ Processed: AnnualCrop\AnnualCrop_2113.jpg
tensor(0.7582, device='cuda:0')
tensor(193.3357, device='cuda:0')
✓ Processed: AnnualCrop\AnnualCrop_2114.jpg
tensor(0.9711, device='cuda:0')
tensor(247.6194, device='cuda:0')
✓ Processed: AnnualCrop\AnnualCrop_2115.jpg
tensor(0.7269, device='cuda:0')
tensor(185.3525, device='cuda:0')
✓ Processed: AnnualCrop\AnnualCrop_2116.jpg
tensor(0.6088, devic

C:\Users\khahung.nguyen\AppData\Local\Temp\ipykernel_14952\1591684577.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path, 

tensor(0.5268, device='cuda:0')
tensor(134.3276, device='cuda:0')
✓ Processed: AnnualCrop\AnnualCrop_2120.jpg
✅ All images processed.
